### Singleton Design pattern 

In [1]:
import os
import re
import numpy as np
import faiss
import torch
from tqdm import tqdm
from collections import Counter
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from llm import LocalLLM


##############################################
# 0. OpenAI Client
##############################################

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"   # Required by the OpenAI client, but Ollama ignores it
)

QWEN_MODEL ="qwen3.6:latest"



##############################################
# 1. Semantic Processor
##############################################

def semantic_puml_processor(text):
    """
    Extracts meaningful tokens from PlantUML:
    - Design pattern roles
    - Class/interface names
    - Relationships
    """

    tokens = []

    # Extract DP roles
    roles = re.findall(r"'\s*DP role:\s*(\w+)", text, re.IGNORECASE)
    tokens.extend([f"role_{r.lower()}" for r in roles])

    # Extract class/interface names
    classes = re.findall(r"(?:class|interface)\s+([\w\.]+)", text)
    for c in classes:
        tokens.append(c.split('.')[-1].lower())

    # Map relationships
    rel_map = {
        r"<\|--": "inheritance",
        r"<\|..": "implementation",
        r"-->": "association",
        r"\*--": "composition"
    }

    for symbol, word in rel_map.items():
        if re.search(symbol, text):
            tokens.append(word)

    return " ".join(tokens)


##############################################
# 2. Document Loading (NO CHUNKING)
##############################################

def load_and_process_docs(folder_path):
    """
    Loads full PlantUML files (no chunking).
    """

    documents = []

    for file in os.listdir(folder_path):

        if file.endswith((".puml", ".plantuml")):

            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:

                raw_content = f.read()
                semantic_content = semantic_puml_processor(raw_content)

                if semantic_content.strip():
                    documents.append({
                        "id": file,
                        "content": semantic_content,
                        "raw": raw_content
                    })

    return documents


##############################################
# 3. Embedding Model
##############################################

class EmbeddingModel:

    def __init__(self, model_name="BAAI/bge-base-en"):

        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.model = SentenceTransformer(model_name).to(self.device)

    def encode(self, texts, batch_size=16):

        return self.model.encode(
            texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        )


##############################################
# 4. Vector Store (FAISS)
##############################################

class VectorStore:

    def __init__(self, embedding_dim):

        self.index = faiss.IndexFlatIP(embedding_dim)
        self.metadata = []

    def add_embeddings(self, embeddings, metadatas):

        self.index.add(embeddings)
        self.metadata.extend(metadatas)

    def search(self, query_embedding, top_k=5):

        distances, indices = self.index.search(query_embedding, top_k)

        results = []

        for i, idx in enumerate(indices[0]):
            if idx != -1:
                res = self.metadata[idx].copy()
                res["score"] = distances[0][i]
                results.append(res)

        return results


##############################################
# 5. LLM Reasoning
##############################################

def llm_reasoning(query_raw, retrieved_docs):

    examples = "\n\n".join(
        [f"Example ({doc['id']}):\n{doc['content']}" for doc in retrieved_docs]
    )

    prompt = f"""

Identify only the design pattern Singleton used in the following PlantUML diagram. 
If no Singleton is found, respond with "nons".

QUERY:
{query_raw}

SIMILAR EXAMPLES:
{examples}


Answer ONLY with (Singleton or nons).
"""

    response = client.chat.completions.create(
        model=QWEN_MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are an expert in GoF software design patterns"
                
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()



##############################################
# 6. Build RAG Index
##############################################

def build_rag_index(folder_path):

    print("Loading PlantUML files...")

    docs = load_and_process_docs(folder_path)

    texts = [doc["content"] for doc in docs]

    embedder = EmbeddingModel()
    embeddings = embedder.encode(texts)

    vector_store = VectorStore(embedding_dim=embeddings.shape[1])
    vector_store.add_embeddings(np.array(embeddings), docs)

    return vector_store, embedder


##############################################
# 7. Pattern Detection
##############################################

def detect_pattern(test_doc, vector_store, embedder, top_k=5, use_llm=True):

    query_text = semantic_puml_processor(test_doc["content"])
    query_embedding = embedder.encode([query_text])

    results = vector_store.search(query_embedding, top_k=top_k)

    # Vector similarity voting
    votes = {}

    for r in results:
        label = r["id"].split('-')[0].split('_')[0]
        votes[label] = votes.get(label, 0) + r["score"]

    predicted_label = max(votes, key=votes.get) if votes else "Unknown"

    # Optional LLM reasoning
    if use_llm:
        llm_prediction = llm_reasoning(test_doc["content"], results)
        return llm_prediction, [r["id"] for r in results]

    return predicted_label, [r["id"] for r in results]


##############################################
# 8. Evaluation
##############################################

def evaluate_detection(test_folder, vector_store, embedder, top_k=5):

    test_docs = load_and_process_docs(test_folder)

    correct_counts = Counter()
    total_counts = Counter()

    for doc in tqdm(test_docs):

        true_label = doc["id"].split('-')[0].split('_')[0]
        total_counts[true_label] += 1

        predicted_label, _ = detect_pattern(
            {"content": doc["raw"]},
            vector_store,
            embedder,
            top_k=top_k
        )

        if predicted_label.lower() == true_label.lower():
            correct_counts[true_label] += 1

        print(f"{doc['id']} | True: {true_label} | Pred: {predicted_label}")

    # Metrics
    per_pattern_accuracy = {
        label: correct_counts[label] / total_counts[label]
        for label in total_counts
    }

    overall_accuracy = sum(correct_counts.values()) / sum(total_counts.values())

    print("\nPer-Pattern Accuracy:")
    for label, acc in per_pattern_accuracy.items():
        print(f"{label}: {acc:.4f}")

    print(f"\nOverall Accuracy: {overall_accuracy:.4f}")

    return overall_accuracy, per_pattern_accuracy

/volatile/home/ya279835/Projets/perso/P-MART-RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ===============================
# 0. Set your OpenAI API Key
# ===============================
import openai
openai.api_key = os.getenv("OPENAI_API_KEY")  # Or set it directly: openai.api_key = "YOUR_KEY"

# ===============================
# 1. Build the RAG index
# ===============================
data_folder = "PMART_PUML/knowledge_base"
vector_store, embedder = build_rag_index(data_folder)



# ===============================
# 2. Evaluate accuracy on a test folder
# ===============================
test_folder = "PMART_PUML/puml_test/singleton"
overall_acc, per_pattern_acc = evaluate_detection(
    test_folder,
    vector_store,
    embedder,
    top_k=5
)

Loading PlantUML files...


  1%|          | 1/93 [00:42<1:04:55, 42.34s/it]

nons_56.puml | True: nons | Pred: nons


  2%|▏         | 2/93 [01:07<48:53, 32.23s/it]  

singleton_21.puml | True: singleton | Pred: nons


...

 66%|██████▌   | 61/93 [36:12<19:59, 37.47s/it]

nons_30.puml | True: nons | Pred: nons


 67%|██████▋   | 62/93 [36:58<20:39, 39.98s/it]

nons_52.puml | True: nons | Pred: nons


Batches: 100%|██████████| 1/1 [00:00<00:00, 131.72it/s]


## Builder Design Pattern

In [1]:
import os
import re
import numpy as np
import faiss
import torch
from tqdm import tqdm
from collections import Counter
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from llm import LocalLLM


##############################################
# 0. OpenAI Client
##############################################

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


##############################################
# 1. Semantic Processor
##############################################

def semantic_puml_processor(text):
    """
    Extracts meaningful tokens from PlantUML:
    - Design pattern roles
    - Class/interface names
    - Relationships
    """

    tokens = []

    # Extract DP roles
    roles = re.findall(r"'\s*DP role:\s*(\w+)", text, re.IGNORECASE)
    tokens.extend([f"role_{r.lower()}" for r in roles])

    # Extract class/interface names
    classes = re.findall(r"(?:class|interface)\s+([\w\.]+)", text)
    for c in classes:
        tokens.append(c.split('.')[-1].lower())

    # Map relationships
    rel_map = {
        r"<\|--": "inheritance",
        r"<\|..": "implementation",
        r"-->": "association",
        r"\*--": "composition"
    }

    for symbol, word in rel_map.items():
        if re.search(symbol, text):
            tokens.append(word)

    return " ".join(tokens)


##############################################
# 2. Document Loading (NO CHUNKING)
##############################################

def load_and_process_docs(folder_path):
    """
    Loads full PlantUML files (no chunking).
    """

    documents = []

    for file in os.listdir(folder_path):

        if file.endswith((".puml", ".plantuml")):

            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:

                raw_content = f.read()
                semantic_content = semantic_puml_processor(raw_content)

                if semantic_content.strip():
                    documents.append({
                        "id": file,
                        "content": semantic_content,
                        "raw": raw_content
                    })

    return documents


##############################################
# 3. Embedding Model
##############################################

class EmbeddingModel:

    def __init__(self, model_name="BAAI/bge-base-en"):

        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.model = SentenceTransformer(model_name).to(self.device)

    def encode(self, texts, batch_size=16):

        return self.model.encode(
            texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        )


##############################################
# 4. Vector Store (FAISS)
##############################################

class VectorStore:

    def __init__(self, embedding_dim):

        self.index = faiss.IndexFlatIP(embedding_dim)
        self.metadata = []

    def add_embeddings(self, embeddings, metadatas):

        self.index.add(embeddings)
        self.metadata.extend(metadatas)

    def search(self, query_embedding, top_k=5):

        distances, indices = self.index.search(query_embedding, top_k)

        results = []

        for i, idx in enumerate(indices[0]):
            if idx != -1:
                res = self.metadata[idx].copy()
                res["score"] = distances[0][i]
                results.append(res)

        return results


##############################################
# 5. LLM Reasoning
##############################################

def llm_reasoning(query_raw, retrieved_docs):

    examples = "\n\n".join(
        [f"Example ({doc['id']}):\n{doc['content']}" for doc in retrieved_docs]
    )

    prompt = f"""

Identify only the design pattern Builder used in the following PlantUML diagram. 
If no builder is found, respond with "nonb".

QUERY:
{query_raw}

SIMILAR EXAMPLES:
{examples}


Answer ONLY with builder or nonb.
"""

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": "You are an expert in GoF software design patterns."},
            {"role": "user", "content": prompt}
        ],
    )

    return response.choices[0].message.content.strip()



##############################################
# 6. Build RAG Index
##############################################

def build_rag_index(folder_path):

    print("Loading PlantUML files...")

    docs = load_and_process_docs(folder_path)

    texts = [doc["content"] for doc in docs]

    embedder = EmbeddingModel()
    embeddings = embedder.encode(texts)

    vector_store = VectorStore(embedding_dim=embeddings.shape[1])
    vector_store.add_embeddings(np.array(embeddings), docs)

    return vector_store, embedder


##############################################
# 7. Pattern Detection
##############################################

def detect_pattern(test_doc, vector_store, embedder, top_k=5, use_llm=True):

    query_text = semantic_puml_processor(test_doc["content"])
    query_embedding = embedder.encode([query_text])

    results = vector_store.search(query_embedding, top_k=top_k)

    # Vector similarity voting
    votes = {}

    for r in results:
        label = r["id"].split('-')[0].split('_')[0]
        votes[label] = votes.get(label, 0) + r["score"]

    predicted_label = max(votes, key=votes.get) if votes else "Unknown"

    # Optional LLM reasoning
    if use_llm:
        llm_prediction = llm_reasoning(test_doc["content"], results)
        return llm_prediction, [r["id"] for r in results]

    return predicted_label, [r["id"] for r in results]


##############################################
# 8. Evaluation
##############################################

def evaluate_detection(test_folder, vector_store, embedder, top_k=5):

    test_docs = load_and_process_docs(test_folder)

    correct_counts = Counter()
    total_counts = Counter()

    for doc in tqdm(test_docs):

        true_label = doc["id"].split('-')[0].split('_')[0]
        total_counts[true_label] += 1

        predicted_label, _ = detect_pattern(
            {"content": doc["raw"]},
            vector_store,
            embedder,
            top_k=top_k
        )

        if predicted_label.lower() == true_label.lower():
            correct_counts[true_label] += 1

        print(f"{doc['id']} | True: {true_label} | Pred: {predicted_label}")

    # Metrics
    per_pattern_accuracy = {
        label: correct_counts[label] / total_counts[label]
        for label in total_counts
    }

    overall_accuracy = sum(correct_counts.values()) / sum(total_counts.values())

    print("\nPer-Pattern Accuracy:")
    for label, acc in per_pattern_accuracy.items():
        print(f"{label}: {acc:.4f}")

    print(f"\nOverall Accuracy: {overall_accuracy:.4f}")

    return overall_accuracy, per_pattern_accuracy

/volatile/home/ya279835/Projets/perso/P-MART-RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===============================
# 0. Set your OpenAI API Key
# ===============================
import openai
openai.api_key = os.getenv("OPENAI_API_KEY")  # Or set it directly: openai.api_key = "YOUR_KEY"

# ===============================
# 1. Build the RAG index
# ===============================
data_folder = "PMART_PUML/knowledge_base"
vector_store, embedder = build_rag_index(data_folder)



# ===============================
# 2. Evaluate accuracy on a test folder
# ===============================
test_folder = "PMART_PUML/puml_test/builder"
overall_acc, per_pattern_acc = evaluate_detection(
    test_folder,
    vector_store,
    embedder,
    top_k=5
)

Loading PlantUML files...


  1%|          | 1/143 [00:02<05:35,  2.36s/it]

nonb_87.puml | True: nonb | Pred: nonb


  1%|▏         | 2/143 [00:03<04:28,  1.91s/it]

nonb_76.puml | True: nonb | Pred: nonb


...

nonb_96.puml | True: nonb | Pred: nonb


 99%|█████████▉| 142/143 [04:22<00:01,  1.75s/it]

nonb_3.puml | True: nonb | Pred: nonb


100%|██████████| 143/143 [04:24<00:00,  1.85s/it]

nonb_108.puml | True: nonb | Pred: nonb

Per-Pattern Accuracy:
nonb: 1.0000
builder: 0.3333

Overall Accuracy: 0.9580


## Abstract Factory design pattern


In [2]:
import os
import re
import numpy as np
import faiss
import torch
from tqdm import tqdm
from collections import Counter
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from llm import LocalLLM


##############################################
# 0. OpenAI Client
##############################################

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


##############################################
# 1. Semantic Processor
##############################################

def semantic_puml_processor(text):
    """
    Extracts meaningful tokens from PlantUML:
    - Design pattern roles
    - Class/interface names
    - Relationships
    """

    tokens = []

    # Extract DP roles
    roles = re.findall(r"'\s*DP role:\s*(\w+)", text, re.IGNORECASE)
    tokens.extend([f"role_{r.lower()}" for r in roles])

    # Extract class/interface names
    classes = re.findall(r"(?:class|interface)\s+([\w\.]+)", text)
    for c in classes:
        tokens.append(c.split('.')[-1].lower())

    # Map relationships
    rel_map = {
        r"<\|--": "inheritance",
        r"<\|..": "implementation",
        r"-->": "association",
        r"\*--": "composition"
    }

    for symbol, word in rel_map.items():
        if re.search(symbol, text):
            tokens.append(word)

    return " ".join(tokens)


##############################################
# 2. Document Loading (NO CHUNKING)
##############################################

def load_and_process_docs(folder_path):
    """
    Loads full PlantUML files (no chunking).
    """

    documents = []

    for file in os.listdir(folder_path):

        if file.endswith((".puml", ".plantuml")):

            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:

                raw_content = f.read()
                semantic_content = semantic_puml_processor(raw_content)

                if semantic_content.strip():
                    documents.append({
                        "id": file,
                        "content": semantic_content,
                        "raw": raw_content
                    })

    return documents


##############################################
# 3. Embedding Model
##############################################

class EmbeddingModel:

    def __init__(self, model_name="BAAI/bge-base-en"):

        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.model = SentenceTransformer(model_name).to(self.device)

    def encode(self, texts, batch_size=16):

        return self.model.encode(
            texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        )


##############################################
# 4. Vector Store (FAISS)
##############################################

class VectorStore:

    def __init__(self, embedding_dim):

        self.index = faiss.IndexFlatIP(embedding_dim)
        self.metadata = []

    def add_embeddings(self, embeddings, metadatas):

        self.index.add(embeddings)
        self.metadata.extend(metadatas)

    def search(self, query_embedding, top_k=5):

        distances, indices = self.index.search(query_embedding, top_k)

        results = []

        for i, idx in enumerate(indices[0]):
            if idx != -1:
                res = self.metadata[idx].copy()
                res["score"] = distances[0][i]
                results.append(res)

        return results


##############################################
# 5. LLM Reasoning
##############################################

def llm_reasoning(query_raw, retrieved_docs):

    examples = "\n\n".join(
        [f"Example ({doc['id']}):\n{doc['content']}" for doc in retrieved_docs]
    )

    prompt = f"""

Identify only the design pattern Abstract Factory used in the following PlantUML diagram. 
If no Abstract Factory is found, respond with "nonab".

QUERY:
{query_raw}

SIMILAR EXAMPLES:
{examples}


Answer ONLY with abstractfactory or nonab.
"""

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": "You are an expert in GoF software design patterns."},
            {"role": "user", "content": prompt}
        ],
    )

    return response.choices[0].message.content.strip()



##############################################
# 6. Build RAG Index
##############################################

def build_rag_index(folder_path):

    print("Loading PlantUML files...")

    docs = load_and_process_docs(folder_path)

    texts = [doc["content"] for doc in docs]

    embedder = EmbeddingModel()
    embeddings = embedder.encode(texts)

    vector_store = VectorStore(embedding_dim=embeddings.shape[1])
    vector_store.add_embeddings(np.array(embeddings), docs)

    return vector_store, embedder


##############################################
# 7. Pattern Detection
##############################################

def detect_pattern(test_doc, vector_store, embedder, top_k=5, use_llm=True):

    query_text = semantic_puml_processor(test_doc["content"])
    query_embedding = embedder.encode([query_text])

    results = vector_store.search(query_embedding, top_k=top_k)

    # Vector similarity voting
    votes = {}

    for r in results:
        label = r["id"].split('-')[0].split('_')[0]
        votes[label] = votes.get(label, 0) + r["score"]

    predicted_label = max(votes, key=votes.get) if votes else "Unknown"

    # Optional LLM reasoning
    if use_llm:
        llm_prediction = llm_reasoning(test_doc["content"], results)
        return llm_prediction, [r["id"] for r in results]

    return predicted_label, [r["id"] for r in results]


##############################################
# 8. Evaluation
##############################################

def evaluate_detection(test_folder, vector_store, embedder, top_k=5):

    test_docs = load_and_process_docs(test_folder)

    correct_counts = Counter()
    total_counts = Counter()

    for doc in tqdm(test_docs):

        true_label = doc["id"].split('-')[0].split('_')[0]
        total_counts[true_label] += 1

        predicted_label, _ = detect_pattern(
            {"content": doc["raw"]},
            vector_store,
            embedder,
            top_k=top_k
        )

        if predicted_label.lower() == true_label.lower():
            correct_counts[true_label] += 1

        print(f"{doc['id']} | True: {true_label} | Pred: {predicted_label}")

    # Metrics
    per_pattern_accuracy = {
        label: correct_counts[label] / total_counts[label]
        for label in total_counts
    }

    overall_accuracy = sum(correct_counts.values()) / sum(total_counts.values())

    print("\nPer-Pattern Accuracy:")
    for label, acc in per_pattern_accuracy.items():
        print(f"{label}: {acc:.4f}")

    print(f"\nOverall Accuracy: {overall_accuracy:.4f}")

    return overall_accuracy, per_pattern_accuracy

/volatile/home/ya279835/Projets/perso/P-MART-RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ===============================
# 0. Set your OpenAI API Key
# ===============================
import openai
openai.api_key = os.getenv("OPENAI_API_KEY")  # Or set it directly: openai.api_key = "YOUR_KEY"

# ===============================
# 1. Build the RAG index
# ===============================
data_folder = "PMART_PUML/knowledge_base"
vector_store, embedder = build_rag_index(data_folder)



# ===============================
# 2. Evaluate accuracy on a test folder
# ===============================
test_folder = "PMART_PUML/puml_test/abstractfactory"
overall_acc, per_pattern_acc = evaluate_detection(
    test_folder,
    vector_store,
    embedder,
    top_k=5
)

Loading PlantUML files...


  1%|          | 1/103 [00:05<10:03,  5.92s/it]

nonab_79.puml | True: nonab | Pred: nonab


  2%|▏         | 2/103 [00:07<05:47,  3.44s/it]

nonab_82.puml | True: nonab | Pred: nonab


...

nonab_67.puml | True: nonab | Pred: nonab


 99%|█████████▉| 102/103 [03:53<00:01,  1.85s/it]

nonab_45.puml | True: nonab | Pred: nonab


100%|██████████| 103/103 [03:55<00:00,  2.28s/it]

nonab_87.puml | True: nonab | Pred: nonab

Per-Pattern Accuracy:
nonab: 0.9885
abstractfactory: 0.7500

Overall Accuracy: 0.9515


## Factory Method Design pattern


In [5]:
import os
import re
import numpy as np
import faiss
import torch
from tqdm import tqdm
from collections import Counter
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from llm import LocalLLM


##############################################
# 0. OpenAI Client
##############################################

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


##############################################
# 1. Semantic Processor
##############################################

def semantic_puml_processor(text):
    """
    Extracts meaningful tokens from PlantUML:
    - Design pattern roles
    - Class/interface names
    - Relationships
    """

    tokens = []

    # Extract DP roles
    roles = re.findall(r"'\s*DP role:\s*(\w+)", text, re.IGNORECASE)
    tokens.extend([f"role_{r.lower()}" for r in roles])

    # Extract class/interface names
    classes = re.findall(r"(?:class|interface)\s+([\w\.]+)", text)
    for c in classes:
        tokens.append(c.split('.')[-1].lower())

    # Map relationships
    rel_map = {
        r"<\|--": "inheritance",
        r"<\|..": "implementation",
        r"-->": "association",
        r"\*--": "composition"
    }

    for symbol, word in rel_map.items():
        if re.search(symbol, text):
            tokens.append(word)

    return " ".join(tokens)


##############################################
# 2. Document Loading (NO CHUNKING)
##############################################

def load_and_process_docs(folder_path):
    """
    Loads full PlantUML files (no chunking).
    """

    documents = []

    for file in os.listdir(folder_path):

        if file.endswith((".puml", ".plantuml")):

            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:

                raw_content = f.read()
                semantic_content = semantic_puml_processor(raw_content)

                if semantic_content.strip():
                    documents.append({
                        "id": file,
                        "content": semantic_content,
                        "raw": raw_content
                    })

    return documents


##############################################
# 3. Embedding Model
##############################################

class EmbeddingModel:

    def __init__(self, model_name="BAAI/bge-base-en"):

        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.model = SentenceTransformer(model_name).to(self.device)

    def encode(self, texts, batch_size=16):

        return self.model.encode(
            texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        )


##############################################
# 4. Vector Store (FAISS)
##############################################

class VectorStore:

    def __init__(self, embedding_dim):

        self.index = faiss.IndexFlatIP(embedding_dim)
        self.metadata = []

    def add_embeddings(self, embeddings, metadatas):

        self.index.add(embeddings)
        self.metadata.extend(metadatas)

    def search(self, query_embedding, top_k=5):

        distances, indices = self.index.search(query_embedding, top_k)

        results = []

        for i, idx in enumerate(indices[0]):
            if idx != -1:
                res = self.metadata[idx].copy()
                res["score"] = distances[0][i]
                results.append(res)

        return results


##############################################
# 5. LLM Reasoning
##############################################

def llm_reasoning(query_raw, retrieved_docs):

    examples = "\n\n".join(
        [f"Example ({doc['id']}):\n{doc['content']}" for doc in retrieved_docs]
    )

    prompt = f"""

Identify only the design pattern Factory Method used in the following PlantUML diagram. 
If no Factory Method is found, respond with "nonfm".

QUERY:
{query_raw}

SIMILAR EXAMPLES:
{examples}


Answer ONLY with factorymethod or nonfm.
"""

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": "You are an expert in GoF software design patterns."},
            {"role": "user", "content": prompt}
        ],
    )

    return response.choices[0].message.content.strip()



##############################################
# 6. Build RAG Index
##############################################

def build_rag_index(folder_path):

    print("Loading PlantUML files...")

    docs = load_and_process_docs(folder_path)

    texts = [doc["content"] for doc in docs]

    embedder = EmbeddingModel()
    embeddings = embedder.encode(texts)

    vector_store = VectorStore(embedding_dim=embeddings.shape[1])
    vector_store.add_embeddings(np.array(embeddings), docs)

    return vector_store, embedder


##############################################
# 7. Pattern Detection
##############################################

def detect_pattern(test_doc, vector_store, embedder, top_k=5, use_llm=True):

    query_text = semantic_puml_processor(test_doc["content"])
    query_embedding = embedder.encode([query_text])

    results = vector_store.search(query_embedding, top_k=top_k)

    # Vector similarity voting
    votes = {}

    for r in results:
        label = r["id"].split('-')[0].split('_')[0]
        votes[label] = votes.get(label, 0) + r["score"]

    predicted_label = max(votes, key=votes.get) if votes else "Unknown"

    # Optional LLM reasoning
    if use_llm:
        llm_prediction = llm_reasoning(test_doc["content"], results)
        return llm_prediction, [r["id"] for r in results]

    return predicted_label, [r["id"] for r in results]


##############################################
# 8. Evaluation
##############################################

def evaluate_detection(test_folder, vector_store, embedder, top_k=5):

    test_docs = load_and_process_docs(test_folder)

    correct_counts = Counter()
    total_counts = Counter()

    for doc in tqdm(test_docs):

        true_label = doc["id"].split('-')[0].split('_')[0]
        total_counts[true_label] += 1

        predicted_label, _ = detect_pattern(
            {"content": doc["raw"]},
            vector_store,
            embedder,
            top_k=top_k
        )

        if predicted_label.lower() == true_label.lower():
            correct_counts[true_label] += 1

        print(f"{doc['id']} | True: {true_label} | Pred: {predicted_label}")

    # Metrics
    per_pattern_accuracy = {
        label: correct_counts[label] / total_counts[label]
        for label in total_counts
    }

    overall_accuracy = sum(correct_counts.values()) / sum(total_counts.values())

    print("\nPer-Pattern Accuracy:")
    for label, acc in per_pattern_accuracy.items():
        print(f"{label}: {acc:.4f}")

    print(f"\nOverall Accuracy: {overall_accuracy:.4f}")

    return overall_accuracy, per_pattern_accuracy

In [7]:
# ===============================
# 0. Set your OpenAI API Key
# ===============================
import openai
openai.api_key = os.getenv("OPENAI_API_KEY")  # Or set it directly: openai.api_key = "YOUR_KEY"

# ===============================
# 1. Build the RAG index
# ===============================
data_folder = "PMART_PUML/knowledge_base"
vector_store, embedder = build_rag_index(data_folder)



# ===============================
# 2. Evaluate accuracy on a test folder
# ===============================
test_folder = "PMART_PUML/puml_test/factorymethod/"
overall_acc, per_pattern_acc = evaluate_detection(
    test_folder,
    vector_store,
    embedder,
    top_k=5
)

Loading PlantUML files...


  1%|          | 1/93 [00:01<02:53,  1.88s/it]

nonfm_6.puml | True: nonfm | Pred: nonfm


  2%|▏         | 2/93 [00:04<03:08,  2.08s/it]

nonfm_64.puml | True: nonfm | Pred: nonfm


...

nonfm_20.puml | True: nonfm | Pred: nonfm


 99%|█████████▉| 92/93 [04:06<00:02,  2.35s/it]

nonfm_66.puml | True: nonfm | Pred: nonfm


100%|██████████| 93/93 [04:14<00:00,  2.74s/it]

nonfm_12.puml | True: nonfm | Pred: nonfm

Per-Pattern Accuracy:
nonfm: 0.8293
factorymethod: 0.7273

Overall Accuracy: 0.8172


## Prototype Design Pattern


In [1]:
import os
import re
import numpy as np
import faiss
import torch
from tqdm import tqdm
from collections import Counter
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from llm import LocalLLM


##############################################
# 0. OpenAI Client
##############################################

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


##############################################
# 1. Semantic Processor
##############################################

def semantic_puml_processor(text):
    """
    Extracts meaningful tokens from PlantUML:
    - Design pattern roles
    - Class/interface names
    - Relationships
    """

    tokens = []

    # Extract DP roles
    roles = re.findall(r"'\s*DP role:\s*(\w+)", text, re.IGNORECASE)
    tokens.extend([f"role_{r.lower()}" for r in roles])

    # Extract class/interface names
    classes = re.findall(r"(?:class|interface)\s+([\w\.]+)", text)
    for c in classes:
        tokens.append(c.split('.')[-1].lower())

    # Map relationships
    rel_map = {
        r"<\|--": "inheritance",
        r"<\|..": "implementation",
        r"-->": "association",
        r"\*--": "composition"
    }

    for symbol, word in rel_map.items():
        if re.search(symbol, text):
            tokens.append(word)

    return " ".join(tokens)


##############################################
# 2. Document Loading (NO CHUNKING)
##############################################

def load_and_process_docs(folder_path):
    """
    Loads full PlantUML files (no chunking).
    """

    documents = []

    for file in os.listdir(folder_path):

        if file.endswith((".puml", ".plantuml")):

            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:

                raw_content = f.read()
                semantic_content = semantic_puml_processor(raw_content)

                if semantic_content.strip():
                    documents.append({
                        "id": file,
                        "content": semantic_content,
                        "raw": raw_content
                    })

    return documents


##############################################
# 3. Embedding Model
##############################################

class EmbeddingModel:

    def __init__(self, model_name="BAAI/bge-base-en"):

        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.model = SentenceTransformer(model_name).to(self.device)

    def encode(self, texts, batch_size=16):

        return self.model.encode(
            texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        )


##############################################
# 4. Vector Store (FAISS)
##############################################

class VectorStore:

    def __init__(self, embedding_dim):

        self.index = faiss.IndexFlatIP(embedding_dim)
        self.metadata = []

    def add_embeddings(self, embeddings, metadatas):

        self.index.add(embeddings)
        self.metadata.extend(metadatas)

    def search(self, query_embedding, top_k=5):

        distances, indices = self.index.search(query_embedding, top_k)

        results = []

        for i, idx in enumerate(indices[0]):
            if idx != -1:
                res = self.metadata[idx].copy()
                res["score"] = distances[0][i]
                results.append(res)

        return results


##############################################
# 5. LLM Reasoning
##############################################

def llm_reasoning(query_raw, retrieved_docs):

    examples = "\n\n".join(
        [f"Example ({doc['id']}):\n{doc['content']}" for doc in retrieved_docs]
    )

    prompt = f"""

Identify only the design pattern Prototype used in the following PlantUML diagram. 
If no Prototype is found, respond with "nonp".

QUERY:
{query_raw}

SIMILAR EXAMPLES:
{examples}


Answer ONLY with prototype or nonp.
"""

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": "You are an expert in GoF software design patterns."},
            {"role": "user", "content": prompt}
        ],
    )

    return response.choices[0].message.content.strip()



##############################################
# 6. Build RAG Index
##############################################

def build_rag_index(folder_path):

    print("Loading PlantUML files...")

    docs = load_and_process_docs(folder_path)

    texts = [doc["content"] for doc in docs]

    embedder = EmbeddingModel()
    embeddings = embedder.encode(texts)

    vector_store = VectorStore(embedding_dim=embeddings.shape[1])
    vector_store.add_embeddings(np.array(embeddings), docs)

    return vector_store, embedder


##############################################
# 7. Pattern Detection
##############################################

def detect_pattern(test_doc, vector_store, embedder, top_k=5, use_llm=True):

    query_text = semantic_puml_processor(test_doc["content"])
    query_embedding = embedder.encode([query_text])

    results = vector_store.search(query_embedding, top_k=top_k)

    # Vector similarity voting
    votes = {}

    for r in results:
        label = r["id"].split('-')[0].split('_')[0]
        votes[label] = votes.get(label, 0) + r["score"]

    predicted_label = max(votes, key=votes.get) if votes else "Unknown"

    # Optional LLM reasoning
    if use_llm:
        llm_prediction = llm_reasoning(test_doc["content"], results)
        return llm_prediction, [r["id"] for r in results]

    return predicted_label, [r["id"] for r in results]


##############################################
# 8. Evaluation
##############################################

def evaluate_detection(test_folder, vector_store, embedder, top_k=5):

    test_docs = load_and_process_docs(test_folder)

    correct_counts = Counter()
    total_counts = Counter()

    for doc in tqdm(test_docs):

        true_label = doc["id"].split('-')[0].split('_')[0]
        total_counts[true_label] += 1

        predicted_label, _ = detect_pattern(
            {"content": doc["raw"]},
            vector_store,
            embedder,
            top_k=top_k
        )

        if predicted_label.lower() == true_label.lower():
            correct_counts[true_label] += 1

        print(f"{doc['id']} | True: {true_label} | Pred: {predicted_label}")

    # Metrics
    per_pattern_accuracy = {
        label: correct_counts[label] / total_counts[label]
        for label in total_counts
    }

    overall_accuracy = sum(correct_counts.values()) / sum(total_counts.values())

    print("\nPer-Pattern Accuracy:")
    for label, acc in per_pattern_accuracy.items():
        print(f"{label}: {acc:.4f}")

    print(f"\nOverall Accuracy: {overall_accuracy:.4f}")

    return overall_accuracy, per_pattern_accuracy

/volatile/home/ya279835/Projets/perso/P-MART-RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===============================
# 0. Set your OpenAI API Key
# ===============================
import openai
openai.api_key = os.getenv("OPENAI_API_KEY")  # Or set it directly: openai.api_key = "YOUR_KEY"

# ===============================
# 1. Build the RAG index
# ===============================
data_folder = "PMART_PUML/knowledge_base"
vector_store, embedder = build_rag_index(data_folder)



# ===============================
# 2. Evaluate accuracy on a test folder
# ===============================
test_folder = "PMART_PUML/puml_test/prototype/"
overall_acc, per_pattern_acc = evaluate_detection(
    test_folder,
    vector_store,
    embedder,
    top_k=5
)

Loading PlantUML files...


  1%|          | 1/93 [00:02<04:14,  2.77s/it]

nonp_56.puml | True: nonp | Pred: nonp


  2%|▏         | 2/93 [00:05<03:55,  2.59s/it]

nonp_59.puml | True: nonp | Pred: nonp


...

prototype_14.puml | True: prototype | Pred: prototype


 99%|█████████▉| 92/93 [03:22<00:02,  2.52s/it]

prototype_5.puml | True: prototype | Pred: nonp


100%|██████████| 93/93 [03:23<00:00,  2.19s/it]

prototype_7.puml | True: prototype | Pred: nonp

Per-Pattern Accuracy:
nonp: 1.0000
prototype: 0.6250

Overall Accuracy: 0.8710
